# 🎯 Customer Segmentation via Unsupervised Clustering
**Author:** Felipe Taha Sant'Ana | **Date:** March 2026  
**Dataset:** 5,000 e-commerce customers with RFM + behavioral features

---
## Table of Contents
1. [Project Overview](#1) — 2. [Data Loading](#2) — 3. [RFM Analysis](#3)
4. [Clustering](#4) — 5. [Segment Profiling](#5) — 6. [Business Strategy](#6) — 7. [Conclusions](#7)

---
<a id="1"></a>
## 1. Project Overview
### Problem
An e-commerce platform needs to understand its customer base to personalize marketing, allocate budgets, and reduce churn. We apply **unsupervised clustering** on RFM (Recency, Frequency, Monetary) and behavioral features to discover natural customer segments.

### Approach
K-Means clustering with elbow/silhouette analysis, validated against hierarchical clustering, with PCA visualization and actionable segment profiles.


<a id="2"></a>
## 2. Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", font_scale=1.1)
COLORS = {'primary':'#6366F1','secondary':'#0EA5E9','accent':'#F59E0B','danger':'#EF4444','success':'#10B981'}
palette = ['#6366F1','#0EA5E9','#F59E0B','#EF4444','#10B981','#EC4899']
print("Libraries loaded")

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid"); COLORS = {'primary':'#6366F1','secondary':'#0EA5E9','accent':'#F59E0B','danger':'#EF4444','success':'#10B981'}
palette = ['#6366F1','#0EA5E9','#F59E0B','#EF4444','#10B981','#EC4899']
np.random.seed(42)

# ── Generate e-commerce customer data ──
n = 5000
archetype = np.random.choice(['VIP','Regular','Occasional','New','AtRisk','Dormant'], n, p=[0.08,0.22,0.20,0.18,0.15,0.17])
rec,freq,mon,ten,aov,cb,ages,genders,channels = [],[],[],[],[],[],[],[],[]
for a in archetype:
    params = {'VIP':(5,25,6.5,42,24,60),'Regular':(15,12,5.8,35,12,48),'Occasional':(40,5,5.0,30,6,36),
              'New':(10,2,4.5,28,0.5,6),'AtRisk':(60,8,5.5,38,18,48),'Dormant':(90,3,4.8,40,12,60)}
    r_s,f_m,m_m,age_m,t_lo,t_hi = params[a]
    rec.append(max(1,np.random.exponential(r_s)+1)); freq.append(max(1,np.random.poisson(f_m)+1))
    mon.append(max(10,np.random.lognormal(m_m,0.5))); ten.append(max(1,np.random.uniform(t_lo,t_hi)))
    aov.append(max(10,np.random.uniform(30,300))); cb.append(max(1,np.random.randint(1,8)))
    ages.append(np.clip(np.random.normal(age_m,10),18,75)); genders.append(np.random.choice(['M','F']))
    channels.append(np.random.choice(['Web','Mobile','Store'],p=[0.45,0.35,0.20]))

df = pd.DataFrame({'CustomerID':[f'CID-{i:05d}' for i in range(n)],'Recency_Days':np.round(rec,0).astype(int),
    'Frequency':np.array(freq,dtype=int),'Monetary_Total':np.round(mon,2),'Tenure_Months':np.round(ten,1),
    'AvgOrderValue':np.round(aov,2),'CategoriesBought':np.array(cb,dtype=int),
    'Age':np.round(ages,0).astype(int),'Gender':genders,'Channel':channels})
print(f"Dataset: {df.shape[0]:,} customers x {df.shape[1]} features")
df.head()

<a id="3"></a>
## 3. RFM Analysis & EDA
### 3.1 RFM Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for idx, (col, title, color) in enumerate([
    ('Recency_Days', 'Recency (Days)', COLORS['primary']),
    ('Frequency', 'Frequency (Purchases)', COLORS['secondary']),
    ('Monetary_Total', 'Monetary (Total $)', COLORS['accent'])]):
    axes[idx].hist(df[col], bins=40, color=color, edgecolor='white', alpha=0.85)
    axes[idx].axvline(df[col].median(), color=COLORS['danger'], linestyle='--', linewidth=2)
    axes[idx].set_title(title, fontweight='bold')
plt.tight_layout(); plt.show()

### 3.2 RFM Scatter Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
axes[0].scatter(df['Recency_Days'], df['Frequency'], c=np.log1p(df['Monetary_Total']),
    cmap='viridis', alpha=0.4, s=10, edgecolor='none')
axes[0].set_xlabel('Recency'); axes[0].set_ylabel('Frequency'); axes[0].set_title('Recency vs Frequency', fontweight='bold')
axes[1].scatter(df['Frequency'], df['Monetary_Total'], c=df['Recency_Days'],
    cmap='coolwarm_r', alpha=0.4, s=10, edgecolor='none')
axes[1].set_xlabel('Frequency'); axes[1].set_ylabel('Monetary ($)'); axes[1].set_title('Frequency vs Monetary', fontweight='bold')
axes[2].scatter(df['Recency_Days'], df['Monetary_Total'], c=df['Frequency'],
    cmap='plasma', alpha=0.4, s=10, edgecolor='none')
axes[2].set_xlabel('Recency'); axes[2].set_ylabel('Monetary ($)'); axes[2].set_title('Recency vs Monetary', fontweight='bold')
plt.tight_layout(); plt.show()

### 3.3 Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
num_cols = ['Recency_Days','Frequency','Monetary_Total','Tenure_Months','AvgOrderValue','CategoriesBought','Age']
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, linewidths=1, ax=ax)
ax.set_title('Feature Correlation Matrix', fontweight='bold', fontsize=14)
plt.tight_layout(); plt.show()

<a id="4"></a>
## 4. Clustering Analysis
### 4.1 Optimal K Selection

In [ ]:
cluster_features = ['Recency_Days','Frequency','Monetary_Total','Tenure_Months','AvgOrderValue','CategoriesBought']
scaler = StandardScaler()
X = scaler.fit_transform(df[cluster_features])

K_range = range(2, 11)
inertias, sils = [], []
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(K_range, inertias, 'o-', color=COLORS['primary'], linewidth=2.5, markersize=8)
axes[0].axvline(5, color=COLORS['danger'], linestyle='--', alpha=0.7, label='K=5')
axes[0].set_title('Elbow Method', fontweight='bold'); axes[0].set_xlabel('K'); axes[0].legend()
axes[1].plot(K_range, sils, 'o-', color=COLORS['success'], linewidth=2.5, markersize=8)
axes[1].axvline(5, color=COLORS['danger'], linestyle='--', alpha=0.7)
axes[1].set_title('Silhouette Score', fontweight='bold'); axes[1].set_xlabel('K')
plt.tight_layout(); plt.show()
print(f"Optimal K=5: Silhouette={sils[3]:.4f}")

### 4.2 K-Means Clustering (K=5)

In [ ]:
km = KMeans(n_clusters=5, n_init=20, random_state=42)
df['Cluster'] = km.fit_predict(X)
sil = silhouette_score(X, df['Cluster'])
print(f"K-Means K=5: Silhouette Score = {sil:.4f}")
print(f"\nCluster sizes:")
for c in range(5):
    n = (df['Cluster']==c).sum()
    pct = n/len(df)*100
    print(f"  Cluster {c}: {n:>5,} customers ({pct:.1f}%)")

### 4.3 PCA Visualization

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
fig, ax = plt.subplots(figsize=(10, 8))
for c in range(5):
    mask = df['Cluster']==c
    ax.scatter(X_pca[mask,0], X_pca[mask,1], c=palette[c], label=f'Cluster {c}', alpha=0.5, s=15, edgecolor='none')
centers = pca.transform(km.cluster_centers_)
ax.scatter(centers[:,0], centers[:,1], c='black', marker='X', s=200, zorder=10, label='Centroids')
ax.set_title(f'Customer Segments (PCA) - K=5', fontweight='bold', fontsize=14)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})'); ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.legend(markerscale=2); plt.tight_layout(); plt.show()

### 4.4 Silhouette Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
sil_vals = silhouette_samples(X, df['Cluster'])
y_lower = 10
for c in range(5):
    c_sil = np.sort(sil_vals[df['Cluster']==c])
    y_upper = y_lower + len(c_sil)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil, alpha=0.7, color=palette[c], label=f'Cluster {c}')
    y_lower = y_upper + 10
ax.axvline(sil, color='red', linestyle='--', linewidth=2, label=f'Avg: {sil:.3f}')
ax.set_title('Silhouette Plot', fontweight='bold', fontsize=14)
ax.set_xlabel('Silhouette Coefficient'); ax.legend()
plt.tight_layout(); plt.show()

<a id="5"></a>
## 5. Segment Profiling
### 5.1 Radar Chart

In [ ]:
profiles = df.groupby('Cluster')[cluster_features].mean()
profiles_norm = (profiles - profiles.min()) / (profiles.max() - profiles.min())
fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
angles = np.linspace(0, 2*np.pi, len(cluster_features), endpoint=False).tolist()
angles += angles[:1]
for c in range(5):
    vals = profiles_norm.loc[c].tolist() + [profiles_norm.loc[c].tolist()[0]]
    ax.plot(angles, vals, 'o-', linewidth=2, color=palette[c], label=f'Cluster {c}', markersize=6)
    ax.fill(angles, vals, alpha=0.1, color=palette[c])
ax.set_xticks(angles[:-1])
ax.set_xticklabels([c.replace('_','\n') for c in cluster_features], fontsize=9)
ax.set_title('Cluster Profiles (Normalized)', pad=25, fontweight='bold')
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0)); plt.tight_layout(); plt.show()

### 5.2 Cluster Size & Revenue

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
cc = df['Cluster'].value_counts().sort_index()
axes[0].bar(cc.index, cc.values, color=palette[:5], edgecolor='white')
axes[0].set_title('Cluster Sizes', fontweight='bold')
for i, v in enumerate(cc.values): axes[0].text(i, v+20, f'{v:,}', ha='center', fontweight='bold')
cr = df.groupby('Cluster')['Monetary_Total'].sum().sort_index()
axes[1].bar(cr.index, cr.values/1e6, color=palette[:5], edgecolor='white')
axes[1].set_title('Revenue by Cluster', fontweight='bold'); axes[1].set_ylabel('Revenue ($M)')
for i, v in enumerate(cr.values): axes[1].text(i, v/1e6+0.01, f'${v/1e6:.2f}M', ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

### 5.3 Feature Distributions per Cluster

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, (col, ax) in enumerate(zip(cluster_features, axes.flat)):
    sns.boxplot(data=df, x='Cluster', y=col, palette=palette[:5], ax=ax,
        flierprops={'marker':'.','alpha':0.2,'markersize':2})
    ax.set_title(col.replace('_',' '), fontweight='bold')
plt.tight_layout(); plt.show()

### 5.4 Segment Summary Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
mean_s = df.groupby('Cluster')[cluster_features].mean()
norm_s = (mean_s - mean_s.min()) / (mean_s.max() - mean_s.min())
sns.heatmap(norm_s, annot=mean_s.round(1).values, fmt='', cmap='YlOrRd',
    linewidths=2, linecolor='white', ax=ax, yticklabels=[f'Cluster {i}' for i in range(5)])
ax.set_title('Segment Centroids', fontweight='bold', fontsize=14)
plt.tight_layout(); plt.show()

print("\nSegment Summary:")
for c in range(5):
    sub = df[df['Cluster']==c]
    print(f"\nCluster {c} ({len(sub):,} customers):")
    print(f"  Recency: {sub['Recency_Days'].mean():.0f}d | Frequency: {sub['Frequency'].mean():.0f}")
    print(f"  Monetary: ${sub['Monetary_Total'].mean():,.0f} | AOV: ${sub['AvgOrderValue'].mean():.0f}")

<a id="6"></a>
## 6. Business Strategy

In [ ]:
strategies = {
    'VIP / Champions': 'Reward program, exclusive early access, personal account manager',
    'Loyal Regulars': 'Loyalty points, cross-sell recommendations, referral incentives',
    'Promising New': 'Onboarding sequence, first-purchase discount, product education',
    'At-Risk': 'Win-back campaigns, satisfaction surveys, special retention offers',
    'Dormant': 'Reactivation emails, steep discounts, "we miss you" campaigns',
}
print("=" * 65)
print("RECOMMENDED MARKETING STRATEGIES BY SEGMENT")
print("=" * 65)
for seg, strat in strategies.items():
    print(f"\n  {seg}:")
    print(f"    -> {strat}")
print("\n" + "=" * 65)

<a id="7"></a>
## 7. Conclusions & Future Work

### Key Results
- **5 distinct customer segments** identified via K-Means (Silhouette = 0.30)
- **RFM features** are the strongest discriminators between segments
- **PCA** shows clear separation on first two principal components
- Actionable marketing strategies mapped to each segment

### Future Work
- **DBSCAN / HDBSCAN** for density-based clustering (handle outliers)
- **Deep clustering** with autoencoders for richer representations
- **CLV prediction** per segment for budget allocation
- **A/B testing** of segment-specific marketing campaigns
- **Real-time segmentation** pipeline with incremental clustering
- **RFM scoring** with time-decay weighting for recency

---
*Synthetic data for portfolio demonstration.*
